# Server Observability (PerfMon + SQLDiagnostics)

This notebook sets up:
- `btris_dbx.observability` schema
- UC Volume storage for raw files
- Delta ingestion for:
  - PerfMon metrics (CSV/Excel)
  - SQLDiagnostics weekly files (one Excel per server per week)

Note: Windows Events ingestion is not used in this version.

## Create schema


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS btris_dbx.observability;

## Set catalog and schema context


In [0]:
%sql
USE CATALOG btris_dbx;
USE SCHEMA observability;

## Define UC Volume base path


In [0]:
BASE_PATH = "/Volumes/btris_dbx/observability/server_observability_vol"
BASE_PATH

'/Volumes/btris_dbx/observability/server_observability_vol'

## Create required folders (PerfMon + SQLDiagnostics)


In [0]:
dbutils.fs.mkdirs(f"{BASE_PATH}/raw/perfmon")
dbutils.fs.mkdirs(f"{BASE_PATH}/raw/sql_diagnostics/inbox")
dbutils.fs.mkdirs(f"{BASE_PATH}/raw/sql_diagnostics/by_server")
display(dbutils.fs.ls(f"{BASE_PATH}/raw"))

path,name,size,modificationTime
dbfs:/Volumes/btris_dbx/observability/server_observability_vol/raw/perfmon/,perfmon/,0,1772264849000
dbfs:/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/,sql_diagnostics/,0,1772264850000


## Create SQLDiagnostics file registry Delta table


In [0]:
%sql
CREATE TABLE IF NOT EXISTS btris_dbx.observability.sql_diagnostics_files_delta (
  server_name STRING,
  snapshot_date DATE,
  file_path STRING,
  inbox_path STRING,
  file_size_bytes BIGINT,
  modified_ts TIMESTAMP,
  ingested_ts TIMESTAMP
)
USING DELTA;

## Preview SQLDiagnostics inbox


In [0]:
display(dbutils.fs.ls(f"{BASE_PATH}/raw/sql_diagnostics/inbox"))

path,name,size,modificationTime
dbfs:/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/inbox/SQLDiagnostics.xlsx,SQLDiagnostics.xlsx,273222,1772264894000
dbfs:/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/inbox/hc1dbsq36pv.xlsx,hc1dbsq36pv.xlsx,269261,1772264894000


## SQLDiagnostics inbox scanner & registry writer


In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

TABLE   = "btris_dbx.observability.sql_diagnostics_files_delta"
INBOX   = f"{BASE_PATH}/raw/sql_diagnostics/inbox"
BY_SVR  = f"{BASE_PATH}/raw/sql_diagnostics/by_server"

def server_from_filename(filename: str) -> str:
    return re.sub(r"\.xlsx$", "", filename, flags=re.IGNORECASE).strip()

items = [x for x in dbutils.fs.ls(INBOX) if x.path.lower().endswith(".xlsx")]

if not items:
    print(f"No .xlsx files found in: {INBOX}")
else:
    rows = []
    for it in items:
        server = server_from_filename(it.name)
        dest_dir  = f"{BY_SVR}/{server}"
        dest_path = f"{dest_dir}/{it.name}"

        dbutils.fs.mkdirs(dest_dir)
        dbutils.fs.cp(it.path, dest_path, True)

        rows.append((server, it.path, dest_path, int(it.size), int(it.modificationTime)))

    schema = StructType([
        StructField("server_name", StringType(), False),
        StructField("inbox_path", StringType(), False),
        StructField("file_path", StringType(), False),
        StructField("file_size_bytes", LongType(), True),
        StructField("modified_time_ms", LongType(), True),
    ])

    df = spark.createDataFrame(rows, schema=schema)

    df = (
        df.withColumn("modified_ts", (F.col("modified_time_ms") / 1000).cast("timestamp"))
          .withColumn("snapshot_date", F.to_date(F.col("modified_ts")))
          .withColumn("ingested_ts", F.current_timestamp())
          .drop("modified_time_ms")
    )

    df.write.mode("append").format("delta").saveAsTable(TABLE)

    display(df.select("server_name","snapshot_date","file_path","modified_ts","ingested_ts").orderBy(F.col("modified_ts").desc()))


server_name,snapshot_date,file_path,modified_ts,ingested_ts
SQLDiagnostics,2026-02-28,/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/SQLDiagnostics/SQLDiagnostics.xlsx,2026-02-28T07:48:14.000Z,2026-02-28T10:22:15.386Z
hc1dbsq36pv,2026-02-28,/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/hc1dbsq36pv/hc1dbsq36pv.xlsx,2026-02-28T07:48:14.000Z,2026-02-28T10:22:15.386Z


## Deduplicate registry (keep latest per server + snapshot_date)


In [0]:
%sql
CREATE OR REPLACE TABLE btris_dbx.observability.sql_diagnostics_files_delta AS
SELECT server_name, snapshot_date, file_path, inbox_path, file_size_bytes, modified_ts, ingested_ts
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY server_name, snapshot_date
           ORDER BY modified_ts DESC, ingested_ts DESC
         ) AS rn
  FROM btris_dbx.observability.sql_diagnostics_files_delta
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


## Verify registry


In [0]:
display(spark.sql("""
SELECT server_name, snapshot_date, file_path, modified_ts
FROM btris_dbx.observability.sql_diagnostics_files_delta
ORDER BY modified_ts DESC
"""))

server_name,snapshot_date,file_path,modified_ts
SQLDiagnostics,2026-02-28,/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/SQLDiagnostics/SQLDiagnostics.xlsx,2026-02-28T07:48:14.000Z
hc1dbsq36pv,2026-02-28,/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/hc1dbsq36pv/hc1dbsq36pv.xlsx,2026-02-28T07:48:14.000Z


## Latest SQLDiagnostics snapshot view (1 row per server)


In [0]:
%sql
CREATE OR REPLACE VIEW btris_dbx.observability.v_latest_sql_diagnostics AS
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY server_name
           ORDER BY snapshot_date DESC, modified_ts DESC
         ) AS rn
  FROM btris_dbx.observability.sql_diagnostics_files_delta
)
WHERE rn = 1;

## Create Bronze table for SQLDiagnostics sheets (generic JSON rows)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS btris_dbx.observability.sql_diagnostics_bronze (
  server_name STRING,
  snapshot_date DATE,
  sheet_name STRING,
  row_json STRING,
  ingested_ts TIMESTAMP
)
USING DELTA;

## Excel dependency (openpyxl)
Install if missing. If already installed, this is a no-op.


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Multi-file, multi-sheet ingestion (first 52 sheets)
Skips empty/NoData sheets; writes idempotently to Bronze.


In [0]:
import json
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType

REGISTRY_VIEW = "btris_dbx.observability.v_latest_sql_diagnostics"   # latest per server
# If you want to ingest *all snapshots*, use the table instead:
# REGISTRY_VIEW = "btris_dbx.observability.sql_diagnostics_files_delta"

BRONZE_TABLE  = "btris_dbx.observability.sql_diagnostics_bronze"

def is_nodata(df: pd.DataFrame) -> bool:
    if df is None or df.empty:
        return True
    txt = df.astype(str).to_string().lower()
    return ("nodata" in txt) or ("no diagnostic data found" in txt)

schema = StructType([
    StructField("server_name", StringType(), False),
    StructField("snapshot_date", DateType(), False),
    StructField("sheet_name", StringType(), False),
    StructField("row_json", StringType(), False),
])

# Pull files to ingest from registry (source of truth)
reg = spark.sql(f"""
SELECT server_name, snapshot_date, file_path
FROM {REGISTRY_VIEW}
ORDER BY snapshot_date DESC, server_name
""").collect()

print(f"Registry rows to ingest: {len(reg)}")

for r in reg:
    server_name   = r["server_name"]
    snapshot_date = r["snapshot_date"]
    file_path     = r["file_path"]  # should be .../raw/sql_diagnostics/by_server/<server>/<file>.xlsx

    # pandas must read /Volumes/... (not dbfs:/Volumes/...)
    vol_path = file_path.replace("dbfs:", "")

    print(f"\n--- Ingesting: server={server_name} | snapshot_date={snapshot_date} | file={vol_path} ---")

    try:
        xls = pd.ExcelFile(vol_path)
        target_sheets = xls.sheet_names[:52]
    except Exception as e:
        print(f"FAILED to open Excel for {server_name}: {e}")
        continue

    out = []
    valid_sheet_count = 0

    for sheet_name in target_sheets:
        try:
            pdf = pd.read_excel(vol_path, sheet_name=sheet_name)

            if is_nodata(pdf):
                continue

            # normalize
            pdf.columns = [str(c).strip() for c in pdf.columns]
            pdf = pdf.dropna(how="all").dropna(axis=1, how="all")
            if pdf.empty:
                continue

            valid_sheet_count += 1

            for rec in pdf.to_dict(orient="records"):
                safe = {str(k): (None if pd.isna(v) else v) for k, v in rec.items()}
                out.append((server_name, snapshot_date, sheet_name, json.dumps(safe, default=str)))

        except Exception as e:
            print(f"  Skipped sheet (error): {sheet_name} -> {e}")

    print(f"Valid sheets: {valid_sheet_count} | Rows prepared: {len(out)}")

    # Idempotent write: remove existing rows for this server+snapshot
    spark.sql(f"""
    DELETE FROM {BRONZE_TABLE}
    WHERE server_name = '{server_name}'
      AND snapshot_date = DATE('{snapshot_date}')
    """)

    if out:
        sdf = spark.createDataFrame(out, schema=schema).withColumn("ingested_ts", F.current_timestamp())
        sdf.write.mode("append").format("delta").saveAsTable(BRONZE_TABLE)

        display(spark.sql(f"""
        SELECT sheet_name, COUNT(*) AS row_count
        FROM {BRONZE_TABLE}
        WHERE server_name = '{server_name}' AND snapshot_date = DATE('{snapshot_date}')
        GROUP BY sheet_name
        ORDER BY row_count DESC
        LIMIT 10
        """))
    else:
        print("No rows to write for this server+snapshot.")

Registry rows to ingest: 2

--- Ingesting: server=SQLDiagnostics | snapshot_date=2026-02-28 | file=/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/SQLDiagnostics/SQLDiagnostics.xlsx ---
Valid sheets: 43 | Rows prepared: 1419


sheet_name,row_count
10-SQL Server Agent Jobs,302
45-CPU Utilization History,256
34-Missing Indexes All Databas,151
4-Configuration Values,97
30-IO Latency by File,63
26-Database Filenames and Path,63
50-Ad hoc Queries,50
46-Top Worker Time Queries,50
51-Top Logical Reads Queries,50
52-Top Avg Elapsed Time Querie,50



--- Ingesting: server=hc1dbsq36pv | snapshot_date=2026-02-28 | file=/Volumes/btris_dbx/observability/server_observability_vol/raw/sql_diagnostics/by_server/hc1dbsq36pv/hc1dbsq36pv.xlsx ---
Valid sheets: 43 | Rows prepared: 2528


sheet_name,row_count
34-Missing Indexes All Databas,1068
10-SQL Server Agent Jobs,468
45-CPU Utilization History,256
4-Configuration Values,97
30-IO Latency by File,60
26-Database Filenames and Path,60
41-Connection Counts by IP Add,56
46-Top Worker Time Queries,50
51-Top Logical Reads Queries,50
50-Ad hoc Queries,50


## Final Bronze verification (rows per server and snapshot)


In [0]:
display(spark.sql("""
SELECT server_name, snapshot_date, COUNT(*) AS bronze_rows
FROM btris_dbx.observability.sql_diagnostics_bronze
GROUP BY server_name, snapshot_date
ORDER BY snapshot_date DESC, server_name
"""))

server_name,snapshot_date,bronze_rows
SQLDiagnostics,2026-02-28,1419
hc1dbsq36pv,2026-02-28,2528


## Latest SQL Diagnostics Files View (App-Facing Contract)

This view provides a single, stable interface for the Streamlit application to retrieve:

- The list of available servers
- The latest snapshot per server
- The file path for downloading the original SQLDiagnostics Excel
- Metadata such as ingestion timestamp and file size

### Purpose

The ingestion pipeline may process multiple snapshots per server over time.  
The application, however, must always reference **only the most recent snapshot** for each server.

This view:

- Ranks snapshots per server using `snapshot_date` and `ingested_ts`
- Selects only the latest row per server
- Ensures deterministic behavior for:
  - Server dropdown population
  - Download Metrics button
  - Health Assessment Report generation

### Source Table

`btris_dbx.observability.sql_diagnostics_files_delta`

### Output View

`btris_dbx.observability.v_sql_diagnostics_latest_files`

This view acts as the contract layer between the ingestion pipeline and the Streamlit application.

In [0]:
%sql
CREATE OR REPLACE VIEW btris_dbx.observability.v_sql_diagnostics_latest_files AS
WITH ranked AS (
  SELECT
    server_name,
    snapshot_date,
    file_path,
    inbox_path,
    file_size_bytes,
    modified_ts,
    ingested_ts,
    ROW_NUMBER() OVER (
      PARTITION BY server_name
      ORDER BY snapshot_date DESC, ingested_ts DESC
    ) AS rn
  FROM btris_dbx.observability.sql_diagnostics_files_delta
)
SELECT
  server_name,
  snapshot_date,
  file_path,
  inbox_path,
  file_size_bytes,
  modified_ts,
  ingested_ts
FROM ranked
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE VIEW btris_dbx.observability.v_sql_diagnostics_latest_bronze AS
WITH latest AS (
    SELECT server_name, snapshot_date
    FROM btris_dbx.observability.v_sql_diagnostics_latest_files
)
SELECT
    b.server_name,
    b.snapshot_date,
    b.sheet_name,
    b.row_json,
    b.ingested_ts
FROM btris_dbx.observability.sql_diagnostics_bronze b
INNER JOIN latest l
    ON b.server_name = l.server_name
   AND b.snapshot_date = l.snapshot_date;

In [0]:
%sql
SELECT
  server_name,
  snapshot_date,
  sheet_name,
  COUNT(*) AS rows_per_sheet
FROM btris_dbx.observability.sql_diagnostics_bronze
GROUP BY server_name, snapshot_date, sheet_name
ORDER BY rows_per_sheet DESC;

server_name,snapshot_date,sheet_name,rows_per_sheet
hc1dbsq36pv,2026-02-28,34-Missing Indexes All Databas,1068
hc1dbsq36pv,2026-02-28,10-SQL Server Agent Jobs,468
SQLDiagnostics,2026-02-28,10-SQL Server Agent Jobs,302
hc1dbsq36pv,2026-02-28,45-CPU Utilization History,256
SQLDiagnostics,2026-02-28,45-CPU Utilization History,256
SQLDiagnostics,2026-02-28,34-Missing Indexes All Databas,151
SQLDiagnostics,2026-02-28,4-Configuration Values,97
hc1dbsq36pv,2026-02-28,4-Configuration Values,97
SQLDiagnostics,2026-02-28,26-Database Filenames and Path,63
SQLDiagnostics,2026-02-28,30-IO Latency by File,63


In [0]:
%sql
SELECT row_json
FROM btris_dbx.observability.sql_diagnostics_bronze
LIMIT 5;

row_json
"{""index_advantage"": 30097.25, ""last_user_seek"": ""2026-02-25 23:24:50"", ""Database.Schema.Table"": ""[EDW20_Prod].[dbo].[FACT_INVESTMENT_TRANSACTION_ALLOCATION]"", ""missing_indexes_for_table"": 35, ""similar_missing_indexes_for_table"": 9, ""equality_columns"": ""[INVESTMENT_VEHICLE_INVESTOR_ID]"", ""inequality_columns"": ""[AS_OF_DATE]"", ""included_columns"": ""[TRANSACTION_TYPE_CODE], [JOURNAL_ENTRY_TYPE_ID], [BATCH_REFERENCE_ID], [INVESTMENT_FUND_KEY], [LE_ALLOCATION_AMOUNT]"", ""user_seeks"": 929, ""avg_total_user_,cost"": 109.67, ""avg_user_impact"": 29.54, ""Short Query Text"": ""SELECT distinct\tCOINVEST_BY_INVESTMENT.TCG_GLOBAL_ID, COINVEST_BY_INVESTMENT.INVESTOR_ID, COINVEST_BY_INVESTMENT.INVESTOR_NAME, COINVEST_BY_INVESTMENT.INVESTMENT_FUND_REPORTING_PREFERRED_NAME, COINVEST_BY_INVESTMENT.REPORT_CLASS, ""}"
"{""index_advantage"": 27495.63, ""last_user_seek"": ""2026-02-26 09:08:31"", ""Database.Schema.Table"": ""[EDW20_Staging_Prod].[dbo].[STG_INVESTRAN_INVESTOR_INVESTOR_ASSOCIATE]"", ""missing_indexes_for_table"": 1, ""similar_missing_indexes_for_table"": 1, ""equality_columns"": null, ""inequality_columns"": ""[DELETED_IND]"", ""included_columns"": ""[INVESTOR_ASSOCIATE_ID], [INVESTOR_ID]"", ""user_seeks"": 666, ""avg_total_user_,cost"": 149.69, ""avg_user_impact"": 27.58, ""Short Query Text"": ""SELECT investor_associate_id, associate_entity_name, associate_first_name, associate_last_name, corporate_ind, r.* FROM (SELECT INVESTOR_INVESTOR_ASSOCIATE.investor_associate_id, INVESTOR_INVESTOR""}"
"{""index_advantage"": 26239.83, ""last_user_seek"": ""2026-02-26 09:17:04"", ""Database.Schema.Table"": ""[EDW20_Prod].[dbo].[Dim_Conform_ILM_Valued_Asset]"", ""missing_indexes_for_table"": 40, ""similar_missing_indexes_for_table"": 1, ""equality_columns"": ""[End_Effective_Date], [Partnership_Operations_Flag]"", ""inequality_columns"": null, ""included_columns"": ""[Valued_Asset_Key], [Begin_Effective_Date], [Inferred_Member], [Valued_Asset_Name], [Valued_Asset_Reporting_Preferred_Name], [Investment_Code], [Investment_Name], [Valuation_Type], [Valued_Asset_Type], [Valued_Asset_Industry], [Country], [Country_Code], [Region], [Region_Code], [ILM_Id], [Deal_Key], [Acquisition_Date], [Exit_Date], [Realization_Status], [Historical_Valued_Asset_Name], [Historical_Valued_Asset_Reporting_Preferred_Name], [Historical_Investment_Code], [Historical_Investment_Name], [Historical_Valuation_Type], [Historical_Valued_Asset_Type], [Historical_Valued_Asset_Industry], [Historical_Country], [Historical_Country_Code], [Historical_Region], [Historical_Region_Code], [Historical_ILM_Id], [Historical_Deal_Key], [Historical_Acquisition_Date], [Historical_Exit_Date], [Historical_Realization_Status], [Historical_Partnership_Operations_Flag], [Created_Datetime], [Last_Updated_Datetime], [Deleted_Ind], [EBX_CODE], [LP_CONNECT_CODE], [LP_CONNECT_NAME], [LP_Connect_Investment_Initial_Deal_Acquisition_Date], [LP_Connect_Investment_Industry_Name], [LP_Connect_Investment_Country_Code], [LP_Connect_Investment_Reported_Exit_Date], [LP_Connect_Investment_Reported_Public_Or_Private_Name], [Investment_Industry], [Country_Of_Operations], [HQ_Country], [HQ_State], [Invest_Industry_AR_CD], [Invest_Industry_TR_CD], [Historical_Invest_Industry_AR_CD], [Historical_Invest_Industry_TR_CD], [isDebt], [Historical_isDebt]"", ""user_seeks"": 118, ""avg_total_user_,cost"": 222.73, ""avg_user_impact"": 99.84, ""Short Query Text"": ""SELECT * FROM DIM_CONFORM_ILM_VALUED_ASSET WHERE End_Effective_Date IS NULL AND PARTNERSHIP_OPERATIONS_FLAG = 'Y'""}"
"{""index_advantage"": 24352.95, ""last_user_seek"": ""2026-02-25 23:24:44"", ""Database.Schema.Table"": ""[EDW20_Prod].[dbo].[FACT_INVESTMENT_TRANSACTION_ALLOCATION]"", ""missing_indexes_for_table"": 35, ""similar_missing_indexes_for_table"": 13, ""equality_columns"": ""[INVESTMENT_FUND_KEY]"", ""inequality_columns"": ""[AS_OF_DATE]"", ""included_columns"": ""[INVESTMENT_VEHICLE_INVESTOR_ID], [TRANSACTION_TYPE_COD